# Swing lows — the mirror test

**Pre-declared, before the first run on real data:**

| | prediction | basis |
|---|---|---|
| hold rate | 27–33% | highs gave 30.0%, band is ±3 points |
| confirmation lag | 1.4–1.7% | highs gave 1.57%; structural, so this checks the code |
| adverse excursion | no prediction | no basis for one |

Structure only. No predictor testing — re-mining eleven predictors in the
other direction on the same data is how a false positive gets found.

## Write `src/pivot_lows.py`

In [1]:
import pathlib

content = r'''"""
Swing lows — the mirror of pivot_auto.find_pivots.

The study tested swing highs only. This asks whether lows behave the same way,
and nothing else: structure only, no predictor testing. Re-mining eleven
predictors on the same data in the other direction is how a false positive gets
found, and the point of this file is a single pre-declared comparison.

PRE-DECLARED, before the first run:
    hold rate         27-33%   (highs gave 30.0%, band is +/- 3 points)
    confirmation lag  1.4-1.7% (highs gave 1.57%; structural, so this is a
                                check on the code rather than on the market)
    adverse excursion no prediction -- no basis for one

Every rule mirrors pivot_auto exactly:
    fractal        low[i] < low[i-1] and low[i] < low[i+1]
    leg            running HIGH since the last qualified low
    qualification  (leg_high - low[i]) >= atr_mult * ATR[i], min_sep bars apart
    span           until a CLOSE below the pivot low, or span_bars
    wicks          count for max_down / max_rise, as in the highs

COLUMN MAPPING against pivot_auto
    max_up   -> max_down   adverse excursion, now BELOW the pivot low
    max_drop -> max_rise   the favourable move, now upward
    gain     -> decline    the leg into the pivot, now a fall
Both adverse columns are positive numbers measuring distance the wrong way.
"""

import numpy as np
import pandas as pd

from pivot_auto import atr

__all__ = ["find_lows", "summarise_lows", "compare"]


def find_lows(df, atr_mult=3.0, atr_len=14, min_sep=2, span_bars=32,
              rise_target=3.5):
    """One row per qualified swing low, with its outcome fields."""
    df = df.reset_index(drop=True).copy()
    df["atr"] = atr(df, atr_len)
    high, low, close = df["high"].values, df["low"].values, df["close"].values
    atrv = df["atr"].values
    n = len(df)

    leg_hi_price, leg_hi_bar, leg_bars, q_lo_bar = np.nan, -1, 0, -1
    out = []
    for i in range(1, n - 1):
        is_fractal = low[i] < low[i - 1] and low[i] < low[i + 1]
        if is_fractal and not np.isnan(atrv[i]):
            sep_ok = q_lo_bar < 0 or (i - q_lo_bar) >= min_sep
            disp_ok = np.isnan(leg_hi_price) or (leg_hi_price - low[i]) >= atr_mult * atrv[i]
            order_ok = leg_hi_bar < 0 or leg_hi_bar < i
            if sep_ok and disp_ok and order_ok and not np.isnan(leg_hi_price):
                ref = low[i]
                decline = (leg_hi_price - ref) / leg_hi_price * 100
                mult = (leg_hi_price - ref) / atrv[i]
                hi_seen, lo_seen = ref, ref
                bars_held, bars_to_35 = np.nan, np.nan
                for k in range(1, span_bars + 1):
                    j = i + k
                    if j >= n:
                        break
                    hi_seen = max(hi_seen, high[j])
                    lo_seen = min(lo_seen, low[j])
                    if np.isnan(bars_to_35) and high[j] >= ref * (1 + rise_target / 100):
                        bars_to_35 = k
                    if close[j] < ref:
                        bars_held = k
                        break
                out.append({
                    "date": df["date"].iloc[i], "bar": i,
                    "atr_mult": round(mult, 2), "decline": round(decline, 2),
                    "bars": leg_bars,
                    "max_down": round(max(ref - lo_seen, 0) / ref * 100, 2),
                    "max_rise": round(max(hi_seen - ref, 0) / ref * 100, 2),
                    "lag_pct": round((close[i + 1] - ref) / ref * 100, 2),
                    "bars_held": bars_held, "bars_to_35": bars_to_35})
                leg_hi_price, leg_hi_bar, leg_bars = np.nan, -1, 0
                q_lo_bar = i
        if np.isnan(leg_hi_price) or high[i] > leg_hi_price:
            leg_hi_price, leg_hi_bar, leg_bars = high[i], i, 0
        else:
            leg_bars += 1
    return pd.DataFrame(out)


def summarise_lows(p, rise_target=3.5):
    """Same shape as pivot_auto.summarise, with the directions flipped."""
    from scipy import stats
    p = p.copy()
    p["held"] = p["bars_held"].isna()
    p["ratio"] = p["max_rise"] / p["decline"]
    p["span"] = p["bars_held"].fillna(32)
    print(f"\n{len(p)} swing lows   {p['date'].min().date()} to {p['date'].max().date()}")
    print("=" * 68)
    print(f"  held the full span: {p['held'].sum()}/{len(p)} ({p['held'].mean()*100:.0f}%)")
    print(f"  reached +{rise_target}%:      {p['bars_to_35'].notna().sum()}/{len(p)} "
          f"({p['bars_to_35'].notna().mean()*100:.0f}%)")
    print("\n  THE TWO POPULATIONS")
    print("  " + "-" * 64)
    print(f"  {'':<12}{'n':>5}{'med rise':>11}{'med ratio':>11}{'med down':>11}{'med decline':>13}")
    for lab, g in [("held", p[p["held"]]), ("failed", p[~p["held"]])]:
        if len(g):
            print(f"  {lab:<12}{len(g):>5}{g['max_rise'].median():>11.2f}"
                  f"{g['ratio'].median():>11.2f}{g['max_down'].median():>11.2f}"
                  f"{g['decline'].median():>13.2f}")
    if "regime" in p.columns:
        print("\n  HOLD RATE BY REGIME")
        print("  " + "-" * 64)
        for r, g in p.groupby("regime"):
            print(f"  {r:<16}{len(g):>5} lows   held {g['held'].mean()*100:>3.0f}%   "
                  f"med rise {g['max_rise'].median():>5.2f}%")
    print("\n  ADVERSE EXCURSION (below the pivot low)")
    print("  " + "-" * 64)
    u = p["max_down"]
    print("  " + "   ".join(f"p{q}: {np.percentile(u, q):.2f}%" for q in (50, 80, 90, 95)))
    print(f"  never traded below the pivot low: {(u == 0).sum()}/{len(p)}")
    print("\n  CONFIRMATION LAG")
    print("  " + "-" * 64)
    print(f"  mean {p['lag_pct'].mean():.2f}%  -- gone before the low is knowable")
    print("\n  SPAN CONFOUND CHECK")
    print("  " + "-" * 64)
    rho, _ = stats.spearmanr(p["span"], p["max_rise"])
    print(f"  span vs max_rise: rho={rho:+.3f}")
    return p


def compare(highs, lows, df):
    """The pre-declared comparison. Prints pass/fail against the two bands.

    `highs` comes from pivot_auto.find_pivots and has no lag column, so the
    confirmation lag is recomputed here from the price data the same way.
    """
    df = df.reset_index(drop=True)
    hi, cl = df["high"].values, df["close"].values
    h_held = highs["bars_held"].isna()
    l_held = lows["bars_held"].isna()
    h_lag = np.mean([(hi[int(b)] - cl[int(b) + 1]) / hi[int(b)] * 100
                     for b in highs["bar"] if int(b) + 1 < len(df)])

    rows = [
        ("n", len(highs), len(lows), ""),
        ("hold rate %", h_held.mean() * 100, l_held.mean() * 100, "27-33"),
        ("adverse p50 %", highs["max_up"].median(), lows["max_down"].median(), "none"),
        ("adverse p80 %", highs["max_up"].quantile(.8), lows["max_down"].quantile(.8), "none"),
        ("med move, held", highs[h_held]["max_drop"].median(),
         lows[l_held]["max_rise"].median(), ""),
        ("med move, failed", highs[~h_held]["max_drop"].median(),
         lows[~l_held]["max_rise"].median(), ""),
        ("confirm lag %", h_lag, lows["lag_pct"].mean(), "1.4-1.7"),
    ]
    print(f"{'':<20}{'highs':>10}{'lows':>10}{'predicted':>12}{'':>8}")
    print("-" * 60)
    for name, a, b, band in rows:
        verdict = ""
        if band == "27-33":
            verdict = "ok" if 27 <= b <= 33 else "MISS"
        elif band == "1.4-1.7":
            verdict = "ok" if 1.4 <= b <= 1.7 else "MISS"
        av = f"{a:>10.2f}" if isinstance(a, float) and np.isfinite(a) else f"{a:>10}"
        print(f"{name:<20}{av}{b:>10.2f}{band:>12}{verdict:>8}")
'''

pathlib.Path('src/pivot_lows.py').write_text(content, encoding='utf-8')
print('wrote src/pivot_lows.py')

wrote src/pivot_lows.py


## Run it

In [2]:
import sys
sys.path.insert(0, 'src')

import pandas as pd
from pivot_auto import find_pivots, add_regime
from pivot_lows import find_lows, summarise_lows, compare

df = pd.read_csv('data/btc_6h.csv', parse_dates=['date'])

highs = find_pivots(df, atr_mult=1.5)
lows  = find_lows(df,   atr_mult=1.5)
lows  = add_regime(lows, df)
lows  = summarise_lows(lows)


622 swing lows   2023-08-01 to 2026-08-28
  held the full span: 233/622 (37%)
  reached +3.5%:      384/622 (62%)

  THE TWO POPULATIONS
  ----------------------------------------------------------------
                  n   med rise  med ratio   med down  med decline
  held          233      10.04       2.37       0.00         3.91
  failed        389       2.99       0.78       1.24         3.54

  HOLD RATE BY REGIME
  ----------------------------------------------------------------
  consolidation     336 lows   held  38%   med rise  4.59%
  down              125 lows   held  36%   med rise  5.20%
  unknown            10 lows   held  50%   med rise  3.30%
  up                151 lows   held  38%   med rise  4.30%

  ADVERSE EXCURSION (below the pivot low)
  ----------------------------------------------------------------
  p50: 0.84%   p80: 1.75%   p90: 2.54%   p95: 3.35%
  never traded below the pivot low: 163/622

  CONFIRMATION LAG
  -------------------------------------------

## The pre-declared comparison

In [3]:
compare(highs, lows, df)

                         highs      lows   predicted        
------------------------------------------------------------
n                          615    622.00                    
hold rate %              29.92     37.46       27-33    MISS
adverse p50 %             0.80      0.84        none        
adverse p80 %             1.78      1.75        none        
med move, held            9.91     10.04                    
med move, failed          2.66      2.99                    
confirm lag %             1.57      1.90     1.4-1.7    MISS


## Record the verdict

Write what happened here, before interpreting it. If hold rate landed
outside 27–33%, that is an asymmetry and worth its own section in
FINDINGS.md. If confirmation lag landed outside 1.4–1.7%, suspect the code
before the market — that quantity is structural.

*(verdict: )*

In [7]:
import numpy as np, pandas as pd

hi, lo, cl = df["high"].values, df["low"].values, df["close"].values

# per-pivot confirmation lag for the highs, computed the same way as the lows
h_lag = pd.Series([(hi[int(b)] - cl[int(b)+1]) / hi[int(b)] * 100
                   for b in highs["bar"] if int(b)+1 < len(df)])
l_lag = lows["lag_pct"]

print("CONFIRMATION LAG -- mean vs median")
print("-" * 52)
print(f"{'':<10}{'mean':>9}{'median':>9}{'p80':>9}{'n':>8}")
for name, s in [("highs", h_lag), ("lows", l_lag)]:
    print(f"{name:<10}{s.mean():>9.2f}{s.median():>9.2f}{s.quantile(.8):>9.2f}{len(s):>8}")

tot = (cl[-1] - cl[0]) / cl[0] * 100
print(f"\nSAMPLE DRIFT")
print("-" * 52)
print(f"  total return over the window: {tot:+.0f}%")

highs = highs.copy(); lows = lows.copy()
if "regime" not in highs.columns:
    highs = add_regime(highs, df)
highs["held"] = highs["bars_held"].isna()
lows["held"]  = lows["bars_held"].isna()
highs["year"] = pd.to_datetime(highs["date"]).dt.year
lows["year"]  = pd.to_datetime(lows["date"]).dt.year

yr_ret = (df.assign(year=df["date"].dt.year)
            .groupby("year")["close"].agg(lambda s: (s.iloc[-1]-s.iloc[0])/s.iloc[0]*100))

print("\nHOLD RATE BY YEAR -- does the gap track that year's return?")
print("-" * 52)
print(f"{'year':<8}{'return':>9}{'highs':>9}{'lows':>9}{'gap':>8}{'n_hi':>7}{'n_lo':>7}")
for y in sorted(set(highs["year"]) & set(lows["year"])):
    a = highs.loc[highs["year"] == y, "held"]
    b = lows.loc[lows["year"] == y, "held"]
    print(f"{y:<8}{yr_ret.get(y, float('nan')):>+9.0f}{a.mean()*100:>9.1f}"
          f"{b.mean()*100:>9.1f}{(b.mean()-a.mean())*100:>+8.1f}{len(a):>7}{len(b):>7}")

print("\nHOLD RATE BY REGIME -- weaker evidence, regime is a label not a return")
print("-" * 52)
print(f"{'regime':<16}{'highs':>9}{'lows':>9}{'gap':>8}{'n_hi':>7}{'n_lo':>7}")
for r in ["up", "consolidation", "down"]:
    a = highs.loc[highs["regime"] == r, "held"]
    b = lows.loc[lows["regime"] == r, "held"]
    if len(a) and len(b):
        print(f"{r:<16}{a.mean()*100:>9.1f}{b.mean()*100:>9.1f}"
              f"{(b.mean()-a.mean())*100:>+8.1f}{len(a):>7}{len(b):>7}")

CONFIRMATION LAG -- mean vs median
----------------------------------------------------
               mean   median      p80       n
highs          1.57     1.27     2.25     615
lows           1.90     1.46     2.74     622

SAMPLE DRIFT
----------------------------------------------------
  total return over the window: +165%

HOLD RATE BY YEAR -- does the gap track that year's return?
----------------------------------------------------
year       return    highs     lows     gap   n_hi   n_lo
2023          +44     23.1     47.3   +24.2     78     91
2024         +121     29.6     38.1    +8.5    206    202
2025           -6     33.0     34.7    +1.7    197    193
2026          -11     29.9     33.8    +4.0    134    136

HOLD RATE BY REGIME -- weaker evidence, regime is a label not a return
----------------------------------------------------
regime              highs     lows     gap   n_hi   n_lo
up                   29.5     37.7    +8.2    193    151
consolidation        30.4 

In [8]:
highs = add_regime(highs, df)
highs["held"] = highs["bars_held"].isna()

print(f"{'regime':<16}{'highs':>9}{'lows':>9}{'gap':>8}{'n_hi':>7}{'n_lo':>7}")
for r in ["up", "consolidation", "down"]:
    a = highs[highs["regime"] == r]["held"]
    b = lows[lows["regime"] == r]["held"]
    if len(a) and len(b):
        print(f"{r:<16}{a.mean()*100:>9.1f}{b.mean()*100:>9.1f}"
              f"{(b.mean()-a.mean())*100:>+8.1f}{len(a):>7}{len(b):>7}")

regime              highs     lows     gap   n_hi   n_lo
up                   29.5     37.7    +8.2    193    151
consolidation        30.4     37.5    +7.1    336    336
down                 25.9     36.0   +10.1     81    125


In [9]:
import numpy as np, pandas as pd
from scipy import stats
from pivot_auto import find_pivots, add_regime
from pivot_lows import find_lows

FLAT_YEARS = [2025, 2026]          # BTC -6% and -11%; the driftless subsample

def _prep(p, df, kind):
    p = add_regime(p.copy(), df)
    p["held"] = p["bars_held"].isna()
    p["year"] = pd.to_datetime(p["date"]).dt.year
    p["adverse"] = p["max_up"] if kind == "high" else p["max_down"]
    p["move"]    = p["max_drop"] if kind == "high" else p["max_rise"]
    return p

def flat(p):
    return p[p["year"].isin(FLAT_YEARS)]

def prop_test(a, b):
    t = [[a.sum(), len(a)-a.sum()], [b.sum(), len(b)-b.sum()]]
    if min(min(r) for r in t) < 1:
        return float("nan")
    return stats.chi2_contingency(t)[1]

print("ATR THRESHOLD SWEEP -- flat years only (2025-2026)")
print("=" * 74)
print(f"{'mult':>6}{'n_hi':>7}{'n_lo':>7}{'hi hold':>10}{'lo hold':>10}"
      f"{'gap':>8}{'p':>8}{'hi adv':>9}{'lo adv':>9}")
print("-" * 74)
for m in (1.5, 2.0, 2.5, 3.0, 4.0):
    h = flat(_prep(find_pivots(df, atr_mult=m), df, "high"))
    l = flat(_prep(find_lows(df,   atr_mult=m), df, "low"))
    if not len(h) or not len(l):
        continue
    g = (l["held"].mean() - h["held"].mean()) * 100
    print(f"{m:>6.1f}{len(h):>7}{len(l):>7}{h['held'].mean()*100:>9.1f}%"
          f"{l['held'].mean()*100:>9.1f}%{g:>+8.1f}{prop_test(h['held'], l['held']):>8.3f}"
          f"{h['adverse'].median():>9.2f}{l['adverse'].median():>9.2f}")

print("\nPIVOT SEPARATION SWEEP -- flat years only, atr_mult=1.5")
print("=" * 74)
print(f"{'min_sep':>8}{'n_hi':>7}{'n_lo':>7}{'hi hold':>10}{'lo hold':>10}"
      f"{'gap':>8}{'p':>8}{'hi adv':>9}{'lo adv':>9}")
print("-" * 74)
for s in (2, 3, 4, 6, 8):
    h = flat(_prep(find_pivots(df, atr_mult=1.5, min_sep=s), df, "high"))
    l = flat(_prep(find_lows(df,   atr_mult=1.5, min_sep=s), df, "low"))
    if not len(h) or not len(l):
        continue
    g = (l["held"].mean() - h["held"].mean()) * 100
    print(f"{s:>8}{len(h):>7}{len(l):>7}{h['held'].mean()*100:>9.1f}%"
          f"{l['held'].mean()*100:>9.1f}%{g:>+8.1f}{prop_test(h['held'], l['held']):>8.3f}"
          f"{h['adverse'].median():>9.2f}{l['adverse'].median():>9.2f}")

h15 = _prep(find_pivots(df, atr_mult=1.5), df, "high")
l15 = _prep(find_lows(df,   atr_mult=1.5), df, "low")

print("\nWITHIN EACH FLAT YEAR SEPARATELY -- atr_mult=1.5")
print("=" * 74)
print(f"{'year':>6}{'n_hi':>7}{'n_lo':>7}{'hi hold':>10}{'lo hold':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
for y in FLAT_YEARS:
    h = h15[h15["year"] == y]; l = l15[l15["year"] == y]
    g = (l["held"].mean() - h["held"].mean()) * 100
    print(f"{y:>6}{len(h):>7}{len(l):>7}{h['held'].mean()*100:>9.1f}%"
          f"{l['held'].mean()*100:>9.1f}%{g:>+8.1f}{prop_test(h['held'], l['held']):>8.3f}")

print("\nBY REGIME -- flat years only")
print("=" * 74)
print(f"{'regime':<16}{'n_hi':>7}{'n_lo':>7}{'hi hold':>10}{'lo hold':>10}{'gap':>8}{'p':>8}")
print("-" * 74)
hf, lf = flat(h15), flat(l15)
for r in ["up", "consolidation", "down"]:
    h = hf[hf["regime"] == r]; l = lf[lf["regime"] == r]
    if len(h) < 5 or len(l) < 5:
        continue
    g = (l["held"].mean() - h["held"].mean()) * 100
    print(f"{r:<16}{len(h):>7}{len(l):>7}{h['held'].mean()*100:>9.1f}%"
          f"{l['held'].mean()*100:>9.1f}%{g:>+8.1f}{prop_test(h['held'], l['held']):>8.3f}")

print("\nTHE TWO LOOSE ENDS -- full sample vs flat years")
print("=" * 74)
for label, hh, ll in [("full sample", h15, l15), ("flat years", hf, lf)]:
    hn = (hh["adverse"] == 0).mean() * 100
    ln = (ll["adverse"] == 0).mean() * 100
    ht = hh["bars_to_35"].notna().mean() * 100
    lt = ll["bars_to_35"].notna().mean() * 100
    print(f"  {label:<14} never adverse: highs {hn:>4.0f}%  lows {ln:>4.0f}%   "
          f"reached 3.5%: highs {ht:>4.0f}%  lows {lt:>4.0f}%")

print("\nADVERSE EXCURSION -- the symmetric one, does it stay symmetric?")
print("=" * 74)
print(f"{'':<14}{'hi p50':>9}{'lo p50':>9}{'hi p80':>9}{'lo p80':>9}{'n_hi':>7}{'n_lo':>7}")
for label, hh, ll in [("full sample", h15, l15), ("flat years", hf, lf)]:
    print(f"  {label:<12}{hh['adverse'].median():>9.2f}{ll['adverse'].median():>9.2f}"
          f"{hh['adverse'].quantile(.8):>9.2f}{ll['adverse'].quantile(.8):>9.2f}"
          f"{len(hh):>7}{len(ll):>7}")

ATR THRESHOLD SWEEP -- flat years only (2025-2026)
  mult   n_hi   n_lo   hi hold   lo hold     gap       p   hi adv   lo adv
--------------------------------------------------------------------------
   1.5    331    329     31.7%     34.3%    +2.6   0.526     0.70     0.80
   2.0    228    212     31.1%     35.8%    +4.7   0.345     0.70     0.89
   2.5    165    154     32.7%     35.7%    +3.0   0.657     0.68     0.86
   3.0    125    115     33.6%     38.3%    +4.7   0.537     0.70     0.86
   4.0     69     66     36.2%     34.8%    -1.4   1.000     0.66     0.82

PIVOT SEPARATION SWEEP -- flat years only, atr_mult=1.5
 min_sep   n_hi   n_lo   hi hold   lo hold     gap       p   hi adv   lo adv
--------------------------------------------------------------------------
       2    331    329     31.7%     34.3%    +2.6   0.526     0.70     0.80
       3    309    304     31.4%     33.9%    +2.5   0.568     0.73     0.81
       4    287    282     32.1%     33.7%    +1.6   0.745   